Gemma

Llama

Deepseek

Basic Prompt
Finetuned Basic Prompt - Health/Science, Politics, Economy
Finetuned Basic Prompt - General Dataset

Finetuning - Health/Science, Politics, Economy
Finetuning - General Dataset


Aggregation Metrics

Decision Agent
Finetuned Decision Agent - General Dataset

Majority Vote
Majority Vote for Finetuned Models

Weighted Majority Vote
Auto-Weight Majority Vote
Autoweight Decision Model + (optional, provide Majority Vote)

Big Models for reference?

 - delete Different Prompts? or maybe leave that because in singlemodel it was visible that it did not made it better?

With Subjects and without subjects?

Paraphrase Statements

Weighted Finetuned?
Change Class labeling (true = true, mostly true and true = true, false and pantsfire = false)

Basic Prompt
Finetuned Basic Prompt - Health/Science, Politics, Economy
Finetuned Basic Prompt - General Dataset

Majority vote — no subjects
Majority vote — with subjects
Decision agent — no subjects
Decision agent — with subjects

FineTuned on general:
Majority vote — no subjects
Majority vote — with subjects
Decision agent — no subjects
Decision agent — with subjects
For fine-tuned experts, the subject setting was fixed per backbone to the configuration that performed best on the LIAR test set in the single-model fine-tuning experiments (Table~\ref{tab:finetune_bestof}). This avoids doubling multi-agent runs while keeping each fine-tuned expert in its strongest baseline configuration

FineTuned on each:
Majority vote — no subjects
Majority vote — with subjects
Decision agent — no subjects
Decision agent — with subjects


Weighted majority with prompted router weights


Paraphrase ablation on the best method (one run)
Big-model reference (one run)

# PreProcessing

In [15]:
import pandas as pd
df = pd.read_csv('../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv', sep='\t')
df

,label,statement,subjects
0,False,Says the Annies List political group supports ...,abortion
1,True,When did the decline of coal start It started ...,"energy,history,job-accomplishments"
2,True,Hillary Clinton agrees with John McCain by vot...,foreign-policy
3,False,Health care reform legislation is likely to ma...,health-care
4,True,The economic turnaround started at the end of ...,"economy,jobs"
...,...,...,...
10235,True,There are a larger number of shark attacks in ...,"animals,elections"
10236,True,Democrats have now become the party of the [At...,elections
10237,True,Says an alternative to Social Security that op...,"retirement,social-security"
10238,False,On lifting the U.S. Cuban embargo and allowing...,"florida,foreign-policy"


In [16]:
import pandas as pd

# Datei laden (anpassen)
df = pd.read_csv('../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv', sep='\t') 

# alle subjects splitten, säubern, unique sammeln
all_subjects = (
    df["subjects"]
    .dropna()
    .astype(str)
    .str.split(",")
    .explode()
    .str.strip()
)

unique_subjects = sorted(all_subjects.unique())

print(f"Anzahl unique subjects: {len(unique_subjects)}")
print(unique_subjects)


Anzahl unique subjects: 142
['10-news-tampa-bay', 'Alcohol', 'abc-news-week', 'abortion', 'afghanistan', 'after-the-fact', 'agriculture', 'animals', 'autism', 'bankruptcy', 'baseball', 'bipartisanship', 'bush-administration', 'campaign-advertising', 'campaign-finance', 'candidates-biography', 'cap-and-trade', 'census', 'children', 'china', 'city-budget', 'city-government', 'civil-rights', 'climate-change', 'colbert-report', 'congress', 'congressional-rules', 'consumer-safety', 'corporations', 'corrections-and-updates', 'county-budget', 'county-government', 'crime', 'criminal-justice', 'death-penalty', 'debates', 'debt', 'deficit', 'disability', 'diversity', 'drugs', 'ebola', 'economy', 'education', 'elections', 'energy', 'environment', 'ethics', 'fake-news', 'families', 'federal-budget', 'financial-regulation', 'fires', 'florida', 'florida-amendments', 'food', 'food-safety', 'foreign-policy', 'gambling', 'gas-prices', 'gays-and-lesbians', 'government-efficiency', 'government-regulation

In [17]:
SUBJECT_TO_SUPER = {
    # --- economy ---
    "agriculture": "economy",
    "bankruptcy": "economy",
    "cap-and-trade": "economy",
    "city-budget": "economy",
    "consumer-safety": "economy",
    "corporations": "economy",
    "county-budget": "economy",
    "debt": "economy",
    "deficit": "economy",
    "economy": "economy",
    "federal-budget": "economy",
    "financial-regulation": "economy",
    "food": "economy",
    "food-safety": "economy",
    "gas-prices": "economy",
    "government-efficiency": "economy",
    "government-regulation": "economy",
    "housing": "economy",
    "income": "economy",
    "infrastructure": "economy",
    "job-accomplishments": "economy",
    "jobs": "economy",
    "labor": "economy",
    "market-regulation": "economy",
    "pensions": "economy",
    "poverty": "economy",
    "retirement": "economy",
    "small-business": "economy",
    "social-security": "economy",
    "state-budget": "economy",
    "state-finances": "economy",
    "stimulus": "economy",
    "taxes": "economy",
    "trade": "economy",
    "transportation": "economy",
    "unions": "economy",
    "wealth": "economy",
    "welfare": "economy",
    "workers": "economy",
    "tourism": "economy",
    "lottery": "economy",
    "gambling": "economy",
    
    # --- health / science
    "Alcohol": "health_science",
    "abortion": "health_science",
    "animals": "health_science",
    "autism": "health_science",
    "children": "health_science",
    "disability": "health_science",
    "diversity": "health_science",
    "drugs": "health_science",
    "ebola": "health_science",
    "families": "health_science",
    "gays-and-lesbians": "health_science",
    "health-care": "health_science",
    "homeless": "health_science",
    "hunger": "health_science",
    "marijuana": "health_science",
    "medicaid": "health_science",
    "medicare": "health_science",
    "public-health": "health_science",
    "science": "health_science",
    "sexuality": "health_science",
    "water": "health_science",
    "weather": "health_science",
    "women": "health_science",
    "fires": "health_science",
    "natural-disasters": "health_science",
    "oil-spill": "health_science",
    "environment": "health_science",
    "climate-change": "health_science",
    "energy": "health_science",
    "nuclear": "health_science",
    "space": "health_science",
    "technology": "health_science",

    # --- politics (inkl. media/misc/rest) ---
    "10-news-tampa-bay": "politics",
    "abc-news-week": "politics",
    "after-the-fact": "politics",
    "afghanistan": "politics",
    "baseball": "politics",
    "bipartisanship": "politics",
    "bush-administration": "politics",
    "campaign-advertising": "politics",
    "campaign-finance": "politics",
    "candidates-biography": "politics",
    "census": "politics",
    "china": "politics",
    "city-government": "politics",
    "civil-rights": "politics",
    "colbert-report": "politics",
    "congress": "politics",
    "congressional-rules": "politics",
    "corrections-and-updates": "politics",
    "county-government": "politics",
    "crime": "politics",
    "criminal-justice": "politics",
    "death-penalty": "politics",
    "debates": "politics",
    "education": "politics",
    "elections": "politics",
    "ethics": "politics",
    "fake-news": "politics",
    "florida": "politics",
    "florida-amendments": "politics",
    "foreign-policy": "politics",
    "guns": "politics",
    "history": "politics",
    "homeland-security": "politics",
    "human-rights": "politics",
    "immigration": "politics",
    "iraq": "politics",
    "islam": "politics",
    "israel": "politics",
    "kagan-nomination": "politics",
    "legal-issues": "politics",
    "marriage": "politics",
    "message-machine": "politics",
    "message-machine-2012": "politics",
    "message-machine-2014": "politics",
    "military": "politics",
    "new-hampshire-2012": "politics",
    "obama-birth-certificate": "politics",
    "occupy-wall-street": "politics",
    "patriotism": "politics",
    "polls": "politics",
    "pop-culture": "politics",
    "population": "politics",
    "privacy": "politics",
    "public-safety": "politics",
    "public-service": "politics",
    "pundits": "politics",
    "recreation": "politics",
    "redistricting": "politics",
    "religion": "politics",
    "sotomayor-nomination": "politics",
    "sports": "politics",
    "states": "politics",
    "supreme-court": "politics",
    "terrorism": "politics",
    "transparency": "politics",
    "urban": "politics",
    "veterans": "politics",
    "voting-record": "politics",
    "county-government": "politics",
}


In [18]:
from collections import Counter

TIE_BREAK = ["economy", "health_science", "politics"]  # bei Gleichstand

def assign_superlabel(subjects_str: str) -> str:
    if subjects_str is None or str(subjects_str).strip() == "":
        return "politics"  # default

    subjects = [s.strip() for s in str(subjects_str).split(",") if s.strip()]
    mapped = [SUBJECT_TO_SUPER.get(s, "politics") for s in subjects]  # unknown -> politics
    counts = Counter(mapped)

    # max count
    max_count = max(counts.values())
    top = [k for k, v in counts.items() if v == max_count]

    if len(top) == 1:
        return top[0]

    # tie-break
    for pref in TIE_BREAK:
        if pref in top:
            return pref
    return "politics"


In [20]:
import pandas as pd

df = pd.read_csv("../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv", sep="\t")
df["superlabel"] = df["subjects"].apply(assign_superlabel)

df[df["superlabel"] == "politics"].to_csv("LIAR/train_politics.csv", index=False)
df[df["superlabel"] == "economy"].to_csv("LIAR/train_economy.csv", index=False)
df[df["superlabel"] == "health_science"].to_csv("LIAR/train_health_science.csv", index=False)

print(df["superlabel"].value_counts())


superlabel
economy           4265
politics          4022
health_science    1953
Name: count, dtype: int64


In [9]:
import pandas as pd

df = pd.read_csv("../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv", sep="\t")
df["superlabel"] = df["subjects"].apply(assign_superlabel)

df[df["superlabel"] == "politics"].to_csv("Datasets/test_politics.csv", index=False)
df[df["superlabel"] == "economy"].to_csv("Datasets/test_economy.csv", index=False)
df[df["superlabel"] == "health_science"].to_csv("Datasets/test_health_science.csv", index=False)

print(df["superlabel"].value_counts())


superlabel
economy           530
politics          505
health_science    232
Name: count, dtype: int64


# MultiAgent Experiments

## Training

In [2]:
import sys
import factcheck_train_lora

2026-01-04 18:03:16.748635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767546196.770339   28229 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767546196.776920   28229 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767546196.794820   28229 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767546196.794841   28229 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767546196.794843   28229 computation_placer.cc:177] computation placer alr

### economy

#### no subjects

In [3]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "google/gemma-7b-it",
    "--train_csv", "LIAR/train_economy.csv",
    "--output_dir", "adapters/gemma_economy_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/214 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,9.466216
20,No log,1.560582
30,7.730700,0.453813
40,7.730700,0.459614
50,0.443900,0.350620
60,0.443900,0.355216
70,0.443900,0.351634
80,0.374800,0.369316
90,0.374800,0.406390
100,0.392200,0.442064


Best checkpoint: adapters/gemma_economy_nosubj/checkpoint-590
Saved BEST adapter to: adapters/gemma_economy_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/gemma_economy_nosubj


In [4]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "meta-llama/Llama-3.1-8B-Instruct",
    "--train_csv", "LIAR/train_economy.csv",
    "--output_dir", "adapters/llama_economy_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,0.934712
20,No log,0.478281
30,1.567700,0.394911
40,1.567700,0.336643
50,0.330300,0.334062
60,0.330300,0.342409
70,0.330300,0.343273
80,0.328800,0.348763
90,0.328800,0.348075
100,0.338300,0.352786


Best checkpoint: adapters/llama_economy_nosubj/checkpoint-630
Saved BEST adapter to: adapters/llama_economy_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/llama_economy_nosubj


In [5]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "deepseek-ai/deepseek-llm-7b-base",
    "--train_csv", "LIAR/train_economy.csv",
    "--output_dir", "adapters/deepseek_economy_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/4051 [00:00<?, ? examples/s]

Map:   0%|          | 0/214 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100001, 'bos_token_id': 100000, 'pad_token_id': 100001}.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,1.886717
20,No log,1.102696
30,1.657400,0.587580
40,1.657400,0.386976
50,0.468100,0.349887
60,0.468100,0.346899
70,0.468100,0.352987
80,0.351100,0.390528
90,0.351100,0.347283
100,0.336900,0.368119


Best checkpoint: adapters/deepseek_economy_nosubj/checkpoint-1410
Saved BEST adapter to: adapters/deepseek_economy_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/deepseek_economy_nosubj


#### with subjects

In [6]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "google/gemma-7b-it",
    "--train_csv", "LIAR/train_economy.csv",
    "--output_dir", "adapters/gemma_economy_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/4051 [00:00<?, ? examples/s]

Map:   0%|          | 0/214 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,10.034568
20,No log,1.834776
30,8.077300,0.415710
40,8.077300,0.372555
50,0.419400,0.428368
60,0.419400,0.357898
70,0.419400,0.449965
80,0.426400,0.377364
90,0.426400,0.359627
100,0.369100,0.361951


Best checkpoint: adapters/gemma_economy_withsubj/checkpoint-910
Saved BEST adapter to: adapters/gemma_economy_withsubj/best_adapter
Saved adapter + tokenizer to: adapters/gemma_economy_withsubj


In [2]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "meta-llama/Llama-3.1-8B-Instruct",
    "--train_csv", "LIAR/train_economy.csv",
    "--output_dir", "adapters/llama_economy_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/4051 [00:00<?, ? examples/s]

Map:   0%|          | 0/214 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,0.912160
20,No log,0.480106
30,1.638200,0.410627
40,1.638200,0.335780
50,0.334900,0.335158
60,0.334900,0.340728
70,0.334900,0.338834
80,0.327000,0.353280
90,0.327000,0.348150
100,0.336600,0.350034


Best checkpoint: adapters/llama_economy_withsubj/checkpoint-640
Saved BEST adapter to: adapters/llama_economy_withsubj/best_adapter
Saved adapter + tokenizer to: adapters/llama_economy_withsubj


In [3]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "deepseek-ai/deepseek-llm-7b-base",
    "--train_csv", "LIAR/train_economy.csv",
    "--output_dir", "adapters/deepseek_economy_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/4051 [00:00<?, ? examples/s]

Map:   0%|          | 0/214 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100001, 'bos_token_id': 100000, 'pad_token_id': 100001}.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,1.681298
20,No log,0.942172
30,1.459300,0.493426
40,1.459300,0.382382
50,0.411900,0.359647
60,0.411900,0.359506
70,0.411900,0.371469
80,0.360100,0.364157
90,0.360100,0.364248
100,0.343000,0.366400


Best checkpoint: adapters/deepseek_economy_withsubj/checkpoint-930
Saved BEST adapter to: adapters/deepseek_economy_withsubj/best_adapter
Saved adapter + tokenizer to: adapters/deepseek_economy_withsubj


### health science

#### no subjects

In [4]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "google/gemma-7b-it",
    "--train_csv", "LIAR/train_health_science.csv",
    "--output_dir", "adapters/gemma_health_science_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Generating train split: 0 examples [00:00, ? examples/s]

Using delimiter: ','


Map:   0%|          | 0/1855 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,8.766231
20,No log,1.395432
30,7.454100,0.346469
40,7.454100,0.360041
50,0.488600,0.420819
60,0.488600,0.338538
70,0.488600,0.340051
80,0.410200,0.344756
90,0.410200,0.374649
100,0.365000,0.365936


Best checkpoint: adapters/gemma_health_science_nosubj/checkpoint-200
Saved BEST adapter to: adapters/gemma_health_science_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/gemma_health_science_nosubj


In [5]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "meta-llama/Llama-3.1-8B-Instruct",
    "--train_csv", "LIAR/train_health_science.csv",
    "--output_dir", "adapters/llama_health_science_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/1855 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,0.842924
20,No log,0.396829
30,1.592200,0.331549
40,1.592200,0.312166
50,0.348300,0.314484
60,0.348300,0.356810
70,0.348300,0.330131
80,0.346300,0.318682
90,0.346300,0.332751
100,0.334600,0.321240


Best checkpoint: adapters/llama_health_science_nosubj/checkpoint-40
Saved BEST adapter to: adapters/llama_health_science_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/llama_health_science_nosubj


In [6]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "deepseek-ai/deepseek-llm-7b-base",
    "--train_csv", "LIAR/train_health_science.csv",
    "--output_dir", "adapters/deepseek_health_science_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/1855 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100001, 'bos_token_id': 100000, 'pad_token_id': 100001}.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,1.873920
20,No log,1.104758
30,1.649700,0.552305
40,1.649700,0.373194
50,0.464000,0.363412
60,0.464000,0.383783
70,0.464000,0.339351
80,0.346500,0.339796
90,0.346500,0.342349
100,0.338100,0.350921


Best checkpoint: adapters/deepseek_health_science_nosubj/checkpoint-650
Saved BEST adapter to: adapters/deepseek_health_science_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/deepseek_health_science_nosubj


#### with subjects

In [7]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "google/gemma-7b-it",
    "--train_csv", "LIAR/train_health_science.csv",
    "--output_dir", "adapters/gemma_health_science_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/1855 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,9.762166
20,No log,1.624654
30,8.138200,0.318837
40,8.138200,0.354409
50,0.442400,0.316803
60,0.442400,0.342224
70,0.442400,0.341795
80,0.430700,0.368630
90,0.430700,0.398523
100,0.368300,0.384719


Best checkpoint: adapters/gemma_health_science_withsubj/checkpoint-560
Saved BEST adapter to: adapters/gemma_health_science_withsubj/best_adapter
Saved adapter + tokenizer to: adapters/gemma_health_science_withsubj


In [3]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "meta-llama/Llama-3.1-8B-Instruct",
    "--train_csv", "LIAR/train_health_science.csv",
    "--output_dir", "adapters/llama_health_science_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/98 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,0.831932
20,No log,0.422104
30,1.573400,0.349478
40,1.573400,0.318300
50,0.346700,0.323761
60,0.346700,0.345947
70,0.346700,0.347250
80,0.336700,0.339309
90,0.336700,0.331738
100,0.327900,0.326838


Best checkpoint: adapters/llama_health_science_withsubj/checkpoint-40
Saved BEST adapter to: adapters/llama_health_science_withsubj/best_adapter
Saved adapter + tokenizer to: adapters/llama_health_science_withsubj


In [4]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "deepseek-ai/deepseek-llm-7b-base",
    "--train_csv", "LIAR/train_health_science.csv",
    "--output_dir", "adapters/deepseek_health_science_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/1855 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 100001, 'bos_token_id': 100000, 'pad_token_id': 100001}.
You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,1.677966
20,No log,0.950544
30,1.469100,0.469710
40,1.469100,0.347650
50,0.416100,0.348347
60,0.416100,0.365867
70,0.416100,0.328623
80,0.344800,0.329901
90,0.344800,0.328354
100,0.340600,0.344779


Best checkpoint: adapters/deepseek_health_science_withsubj/checkpoint-450
Saved BEST adapter to: adapters/deepseek_health_science_withsubj/best_adapter
Saved adapter + tokenizer to: adapters/deepseek_health_science_withsubj


### politics

#### no subjects

In [5]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "google/gemma-7b-it",
    "--train_csv", "LIAR/train_politics.csv",
    "--output_dir", "adapters/gemma_politics_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Generating train split: 0 examples [00:00, ? examples/s]

Using delimiter: ','


Map:   0%|          | 0/3820 [00:00<?, ? examples/s]

Map:   0%|          | 0/202 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,9.117741
20,No log,1.377817
30,7.556600,0.361440
40,7.556600,0.490231
50,0.420100,0.465948
60,0.420100,0.372060
70,0.420100,0.385859
80,0.414000,0.342643
90,0.414000,0.435611
100,0.418000,0.338993


Best checkpoint: adapters/gemma_politics_nosubj/checkpoint-820
Saved BEST adapter to: adapters/gemma_politics_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/gemma_politics_nosubj


In [6]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "meta-llama/Llama-3.1-8B-Instruct",
    "--train_csv", "LIAR/train_politics.csv",
    "--output_dir", "adapters/llama_politics_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/3820 [00:00<?, ? examples/s]

Map:   0%|          | 0/202 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/jovyan/3_Multi_Agent_Experiments/factcheck_train_lora.py:309: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
10,No log,0.927482
20,No log,0.414023
30,1.708400,0.330661
40,1.708400,0.332439
50,0.349300,0.355252
60,0.349300,0.320891
70,0.349300,0.318375
80,0.324200,0.312014
90,0.324200,0.348914
100,0.333400,0.313442


Best checkpoint: adapters/llama_politics_nosubj/checkpoint-800
Saved BEST adapter to: adapters/llama_politics_nosubj/best_adapter
Saved adapter + tokenizer to: adapters/llama_politics_nosubj


In [ ]:
# politics
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "deepseek-ai/deepseek-llm-7b-base",
    "--train_csv", "LIAR/train_politics.csv",
    "--output_dir", "adapters/deepseek_politics_nosubj",
    "--prompt_id", "standard",
]
factcheck_train_lora.main()

Using delimiter: ','


Map:   0%|          | 0/3820 [00:00<?, ? examples/s]

Map:   0%|          | 0/202 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

#### with subjects

In [ ]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "google/gemma-7b-it",
    "--train_csv", "LIAR/train_politics.csv",
    "--output_dir", "adapters/gemma_politics_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

In [ ]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "meta-llama/Llama-3.1-8B-Instruct",
    "--train_csv", "LIAR/train_politics.csv",
    "--output_dir", "adapters/llama_politics_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

In [ ]:
sys.argv = [
    "factcheck_train_lora.py",
    "--model_name", "deepseek-ai/deepseek-llm-7b-base",
    "--train_csv", "LIAR/train_politics.csv",
    "--output_dir", "adapters/deepseek_politics_withsubj",
    "--prompt_id", "standard",
    "--use_subjects",
]
factcheck_train_lora.main()

# Evaluation

## Finetuned - No Subjects - Decision

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/gemma_politics_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/gemma_economy_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/gemma_health_science_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "google/gemma-7b-it",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="google/gemma-7b-it",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/gemma_finetuned_no_subjects.csv",
    use_subjects=False
)
df_res


E0000 00:00:1767803526.458248   32306 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767803526.467713   32306 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767803526.486049   32306 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767803526.486076   32306 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767803526.486079   32306 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767803526.486081   32306 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='adapters/gemma_politics_nosubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   4%|▍         | 51/1267 [10:52<4:19:28, 12.80s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/llama_politics_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/llama_economy_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/llama_health_science_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="meta-llama/Llama-3.1-8B-Instruct",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/llama_finetuned_no_subjects.csv",
    use_subjects=False
)
df_res


E0000 00:00:1767876245.344638   32935 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767876245.362895   32935 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767876245.459629   32935 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767876245.459665   32935 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767876245.459667   32935 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767876245.459669   32935 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='adapters/llama_politics_nosubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 0/1267 [00:00<?, ?statement/s]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/deepseek_politics_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/deepseek_economy_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/deepseek_health_science_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "deepseek-ai/deepseek-llm-7b-base",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="deepseek-ai/deepseek-llm-7b-base",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/deepseek_finetuned_no_subjects.csv",
    use_subjects=False
)
df_res


E0000 00:00:1768139713.501754   35716 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768139713.508921   35716 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768139713.530771   35716 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768139713.530797   35716 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768139713.530799   35716 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768139713.530801   35716 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='adapters/deepseek_politics_nosubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 3/1267 [00:19<2:19:18,  6.61s/statement]

## Finetuned - With Subjects - Decision

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/gemma_politics_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/gemma_economy_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/gemma_health_science_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "google/gemma-7b-it",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="google/gemma-7b-it",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/gemma_finetuned_with_subjects.csv",
    use_subjects=True
)
df_res


E0000 00:00:1767991081.695011   34073 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767991081.702626   34073 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767991081.721951   34073 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767991081.721974   34073 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767991081.721976   34073 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767991081.721978   34073 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='adapters/gemma_politics_withsubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 1/1267 [00:12<4:34:16, 13.00s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/llama_politics_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/llama_economy_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/llama_health_science_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="meta-llama/Llama-3.1-8B-Instruct",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/llama_finetuned_with_subjects.csv",
    use_subjects=True
)
df_res


E0000 00:00:1768056839.680634   34730 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768056839.687724   34730 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768056839.709376   34730 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768056839.709401   34730 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768056839.709404   34730 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768056839.709405   34730 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='adapters/llama_politics_withsubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 0/1267 [00:00<?, ?statement/s]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/deepseek_politics_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/deepseek_economy_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "adapters/deepseek_health_science_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "deepseek-ai/deepseek-llm-7b-base",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="deepseek-ai/deepseek-llm-7b-base",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/deepseek_finetuned_with_subjects.csv",
    use_subjects=True
)
df_res


E0000 00:00:1768189051.323100   36190 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768189051.330336   36190 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768189051.351286   36190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768189051.351308   36190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768189051.351310   36190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768189051.351313   36190 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='adapters/deepseek_politics_withsubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Expert: politics:   1%|          | 8/1267 [00:48<2:07:14,  6.06s/statement]

## General - No Subjects - Decision

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "google/gemma-7b-it",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "google/gemma-7b-it",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "google/gemma-7b-it",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "google/gemma-7b-it",
    "max_new_tokens": 192,
}

df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="google/gemma-7b-it",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/gemma_general_no_subjects.csv",
    use_subjects=False
)
df_res

E0000 00:00:1768243184.323414   36960 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768243184.330394   36960 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768243184.351758   36960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768243184.351785   36960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768243184.351787   36960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768243184.351789   36960 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='google/gemma-7b-it' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 3/1267 [00:40<4:40:09, 13.30s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="meta-llama/Llama-3.1-8B-Instruct",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/llama_general_no_subjects.csv",
    use_subjects=False
)
df_res


E0000 00:00:1768330508.366549   37718 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768330508.373499   37718 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768330508.394405   37718 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768330508.394426   37718 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768330508.394428   37718 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768330508.394430   37718 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='meta-llama/Llama-3.1-8B-Instruct' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   1%|          | 8/1267 [01:08<2:58:00,  8.48s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "deepseek-ai/deepseek-llm-7b-base",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "deepseek-ai/deepseek-llm-7b-base",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "deepseek-ai/deepseek-llm-7b-base",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "deepseek-ai/deepseek-llm-7b-base",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="deepseek-ai/deepseek-llm-7b-base",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/deepseek_general_no_subjects.csv",
    use_subjects=False
)
df_res


E0000 00:00:1768407295.376622   38098 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768407295.383448   38098 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768407295.405404   38098 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768407295.405429   38098 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768407295.405431   38098 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768407295.405434   38098 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='deepseek-ai/deepseek-llm-7b-base' (sequential load)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 2/1267 [00:13<2:18:17,  6.56s/statement]

## General - With subjects - Decision

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "google/gemma-7b-it",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "google/gemma-7b-it",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "google/gemma-7b-it",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "google/gemma-7b-it",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="google/gemma-7b-it",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/gemma_general_with_subjects.csv",
    use_subjects=True
)
df_res


E0000 00:00:1768483603.740112   38927 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768483603.747173   38927 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768483603.769455   38927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768483603.769504   38927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768483603.769506   38927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768483603.769508   38927 computation_placer.cc:177] computation placer already registered. Please check linka


🧠 Running expert 'politics' with model_id='google/gemma-7b-it' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 2/1267 [00:32<5:40:05, 16.13s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="meta-llama/Llama-3.1-8B-Instruct",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/llama_general_with_subjects.csv",
    use_subjects=True
)
df_res



🧠 Running expert 'politics' with model_id='meta-llama/Llama-3.1-8B-Instruct' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics: 100%|██████████| 1267/1267 [4:32:34<00:00, 12.91s/statement]  



🧠 Running expert 'economy' with model_id='meta-llama/Llama-3.1-8B-Instruct' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: economy: 100%|██████████| 1267/1267 [4:07:05<00:00, 11.70s/statement] 



🧠 Running expert 'health_science' with model_id='meta-llama/Llama-3.1-8B-Instruct' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: health_science:   0%|          | 2/1267 [00:29<5:04:50, 14.46s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "deepseek-ai/deepseek-llm-7b-base",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "deepseek-ai/deepseek-llm-7b-base",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "deepseek-ai/deepseek-llm-7b-base",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "deepseek-ai/deepseek-llm-7b-base",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="deepseek-ai/deepseek-llm-7b-base",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/deepseek_general_with_subjects.csv",
    use_subjects=True
)
df_res



🧠 Running expert 'politics' with model_id='deepseek-ai/deepseek-llm-7b-base' (sequential load)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 2/1267 [00:16<2:53:49,  8.24s/statement]

## Liar - No Subject - Decision

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/gemma_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/gemma_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/gemma_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "google/gemma-7b-it",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="google/gemma-7b-it",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/gemma_LIAR_no_subjects.csv",
    use_subjects=False
)
df_res



🧠 Running expert 'politics' with model_id='../2_Finetuning_Masterthesis/adapters/gemma_general_nosubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 1/1267 [00:16<5:54:38, 16.81s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/llama_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/llama_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/llama_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="meta-llama/Llama-3.1-8B-Instruct",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/llama_LIAR_no_subjects.csv",
    use_subjects=False
)
df_res



🧠 Running expert 'politics' with model_id='../2_Finetuning_Masterthesis/adapters/llama_general_nosubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 1/1267 [00:15<5:24:39, 15.39s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/deepseek_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/deepseek_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/deepseek_general_nosubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "deepseek-ai/deepseek-llm-7b-base",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="deepseek-ai/deepseek-llm-7b-base",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/deepseek_LIAR_no_subjects.csv",
    use_subjects=False
)
df_res



🧠 Running expert 'politics' with model_id='../2_Finetuning_Masterthesis/adapters/deepseek_general_nosubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 3/1267 [00:18<2:05:15,  5.95s/statement]

## Liar - With Subjects - Decision

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/gemma_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/gemma_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/gemma_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "google/gemma-7b-it",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="google/gemma-7b-it",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/gemma_LIAR_with_subjects.csv",
    use_subjects=True
)
df_res



🧠 Running expert 'politics' with model_id='../2_Finetuning_Masterthesis/adapters/gemma_general_withsubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 1/1267 [00:14<4:55:54, 14.02s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/llama_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/llama_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/llama_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="meta-llama/Llama-3.1-8B-Instruct",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/llama_LIAR_with_subjects.csv",
    use_subjects=True
)
df_res



🧠 Running expert 'politics' with model_id='../2_Finetuning_Masterthesis/adapters/llama_general_withsubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 1/1267 [00:15<5:23:48, 15.35s/statement]

In [ ]:
from multiagent_factcheck_eval_sequential import process_claims_multi_experts

experts_cfg = [
    {
        "name": "politics",
        "system": (
            "You are a subject-matter expert in U.S. politics, elections, legislation, and public records. "
            "Task: Verify the claim strictly using publicly verifiable sources (official roll calls, bill texts, "
            "state/federal statutes, FEC/CRP filings, governor/Congress records, reputable outlets). If evidence is "
            'insufficient or ambiguous, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/deepseek_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "economy",
        "system": (
            "You are a subject-matter expert in macroeconomics, labor statistics, public budgets, and taxation. "
            "Verify using BLS, BEA, CBO, JCT, Treasury/IRS, OMB, and credible international sources "
            "(IMF/World Bank/WTO). Define metrics and time ranges precisely; if the data is unclear, cherry-picked, "
            'or not comparable, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/deepseek_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
    {
        "name": "health_science",
        "system": (
            "You are a subject-matter expert in healthcare policy, public health, and scientific evidence appraisal. "
            "Verify with CDC, FDA, NIH, CMS, WHO, Cochrane, and peer-reviewed literature (PubMed). Distinguish "
            'correlation vs. causation and specify years/locations; if evidence is weak or conflicting, output "False". '
            'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
            'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
        ),
        "model_id": "../2_Finetuning_Masterthesis/adapters/deepseek_general_withsubj/best_adapter",
        "max_new_tokens": 176,
    },
]

decision_cfg = {
    "name": "final_decision",
    "system": (
        "You are a senior fact-checking adjudicator. You receive multiple expert analyses, "
        "each with a verdict, explanation, and weight.\n\n"
        "Task:\n"
        "- Evaluate the experts' reasoning quality, factual grounding, and agreement with publicly verifiable evidence.\n"
        "- Consider the explicit weights assigned to each expert.\n"
        "- Prioritize well-supported, domain-relevant arguments over unsupported or vague statements.\n"
        "- If the majority view (weighted) is inconclusive or poorly justified, choose 'False'.\n\n"
        'Return ONLY a JSON object with fields: {"verdict":"True|False","explanation":"2-4 concise sentences, '
        'no stepwise reasoning or lists"}. No markdown, no preface, no code fences, no extra keys.'
    ),
    "model_id": "deepseek-ai/deepseek-llm-7b-base",
    "max_new_tokens": 192,
}


df_res = process_claims_multi_experts(
    csv_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
    experts_cfg=experts_cfg,
    nrows=2000,
    default_model_id="deepseek-ai/deepseek-llm-7b-base",
    show_progress=True,
    decision_cfg=decision_cfg,
    save_path="Results/deepseek_LIAR_with_subjects.csv",
    use_subjects=True
)
df_res



🧠 Running expert 'politics' with model_id='../2_Finetuning_Masterthesis/adapters/deepseek_general_withsubj/best_adapter' (sequential load)...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Expert: politics:   0%|          | 2/1267 [00:09<1:40:44,  4.78s/statement]

# weighted routing

## no subj

In [3]:
from weight_routing import generate_router_weights_for_file
router_model_id = "google/gemma-7b-it"

df_out = generate_router_weights_for_file(
        input_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
        router_model_id=router_model_id,
        output_path="Results/router_weights_gemma_withoutsubjects.csv",
        sep="\t",
        statement_col="statement",
        subject_col="subjects",
        use_subjects=False, 
        nrows=2000,
        max_new_tokens=64,
        temperature=0.0,
    )

print(df_out.head(3))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧭 Routing weights: 100%|██████████| 1267/1267 [28:50<00:00,  1.37s/it]


✅ Saved: Results/router_weights_gemma_withoutsubjects.csv
   label                                          statement  \
0   True  Building a wall on the U.S . Mexico border wil...   
1  False  Wisconsin is on pace to double the number of l...   
2  False  Says John McCain has done nothing to help the ...   

                          subjects  w_politics  w_economy  w_health_science  
0                      immigration    0.750000   0.250000               0.0  
1                             jobs    0.571429   0.428571               0.0  
2  military,veterans,voting-record    0.600000   0.200000               0.2  


In [4]:
from weight_routing import generate_router_weights_for_file
router_model_id = "meta-llama/Llama-3.1-8B-Instruct"

df_out = generate_router_weights_for_file(
        input_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
        router_model_id=router_model_id,
        output_path="Results/router_weights_llama_withoutsubjects.csv",
        sep="\t",
        statement_col="statement",
        subject_col="subjects",
        use_subjects=False, 
        nrows=2000,
        max_new_tokens=64,
        temperature=0.0,
    )

print(df_out.head(3))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧭 Routing weights: 100%|██████████| 1267/1267 [24:38<00:00,  1.17s/it]


✅ Saved: Results/router_weights_llama_withoutsubjects.csv
   label                                          statement  \
0   True  Building a wall on the U.S . Mexico border wil...   
1  False  Wisconsin is on pace to double the number of l...   
2  False  Says John McCain has done nothing to help the ...   

                          subjects  w_politics  w_economy  w_health_science  
0                      immigration         0.7        0.2               0.1  
1                             jobs         0.6        0.4               0.0  
2  military,veterans,voting-record         1.0        0.0               0.0  


In [5]:
from weight_routing import generate_router_weights_for_file
router_model_id = "deepseek-ai/deepseek-llm-7b-base"

df_out = generate_router_weights_for_file(
        input_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
        router_model_id=router_model_id,
        output_path="Results/router_weights_deepseek_withoutsubjects.csv",
        sep="\t",
        statement_col="statement",
        subject_col="subjects",
        use_subjects=False,
        nrows=2000,
        max_new_tokens=64,
        temperature=0.0,
    )

print(df_out.head(3))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🧭 Routing weights: 100%|██████████| 1267/1267 [29:52<00:00,  1.41s/it]


✅ Saved: Results/router_weights_deepseek_withoutsubjects.csv
   label                                          statement  \
0   True  Building a wall on the U.S . Mexico border wil...   
1  False  Wisconsin is on pace to double the number of l...   
2  False  Says John McCain has done nothing to help the ...   

                          subjects  w_politics  w_economy  w_health_science  
0                      immigration    0.500000   0.200000          0.300000  
1                             jobs    0.333333   0.333333          0.333333  
2  military,veterans,voting-record    0.500000   0.200000          0.300000  


## subj

In [2]:
from weight_routing import generate_router_weights_for_file
router_model_id = "google/gemma-7b-it"

df_out = generate_router_weights_for_file(
        input_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
        router_model_id=router_model_id,
        output_path="Results/router_weights_gemma_withsubjects.csv",
        sep="\t",
        statement_col="statement",
        subject_col="subjects",
        use_subjects=True,
        nrows=2000,
        max_new_tokens=64,
        temperature=0.0,
    )

print(df_out.head(3))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧭 Routing weights: 100%|██████████| 1267/1267 [27:44<00:00,  1.31s/it]

✅ Saved: Results/router_weights_gemma_withsubjects.csv
   label                                          statement  \
0   True  Building a wall on the U.S . Mexico border wil...   
1  False  Wisconsin is on pace to double the number of l...   
2  False  Says John McCain has done nothing to help the ...   

                          subjects  w_politics  w_economy  w_health_science  
0                      immigration    0.750000   0.250000               0.0  
1                             jobs    0.666667   0.333333               0.0  
2  military,veterans,voting-record    0.750000   0.250000               0.0  


In [6]:
from weight_routing import generate_router_weights_for_file
router_model_id = "meta-llama/Llama-3.1-8B-Instruct"

df_out = generate_router_weights_for_file(
        input_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
        router_model_id=router_model_id,
        output_path="Results/router_weights_llama_withsubjects.csv",
        sep="\t",
        statement_col="statement",
        subject_col="subjects",
        use_subjects=True, 
        nrows=2000,
        max_new_tokens=64,
        temperature=0.0,
    )

print(df_out.head(3))


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧭 Routing weights: 100%|██████████| 1267/1267 [24:28<00:00,  1.16s/it]


✅ Saved: Results/router_weights_llama_withsubjects.csv
   label                                          statement  \
0   True  Building a wall on the U.S . Mexico border wil...   
1  False  Wisconsin is on pace to double the number of l...   
2  False  Says John McCain has done nothing to help the ...   

                          subjects  w_politics  w_economy  w_health_science  
0                      immigration         0.4        0.3               0.3  
1                             jobs         0.4        0.6               0.0  
2  military,veterans,voting-record         0.8        0.1               0.1  


In [7]:
from weight_routing import generate_router_weights_for_file
router_model_id = "deepseek-ai/deepseek-llm-7b-base"

df_out = generate_router_weights_for_file(
        input_path="../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv",
        router_model_id=router_model_id,
        output_path="Results/router_weights_deepseek_withsubjects.csv",
        sep="\t",
        statement_col="statement",
        subject_col="subjects",
        use_subjects=True,
        nrows=2000,
        max_new_tokens=64,
        temperature=0.0,
    )

print(df_out.head(3))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🧭 Routing weights: 100%|██████████| 1267/1267 [30:02<00:00,  1.42s/it]


✅ Saved: Results/router_weights_deepseek_withsubjects.csv
   label                                          statement  \
0   True  Building a wall on the U.S . Mexico border wil...   
1  False  Wisconsin is on pace to double the number of l...   
2  False  Says John McCain has done nothing to help the ...   

                          subjects  w_politics  w_economy  w_health_science  
0                      immigration    0.500000   0.200000          0.300000  
1                             jobs    0.333333   0.333333          0.333333  
2  military,veterans,voting-record    0.500000   0.200000          0.300000  


# Evaluation

In [1]:
from evaluation import evaluate_results_dir

summary_df, details = evaluate_results_dir(
    results_dir="Results",
    output_summary_csv="Results/evaluation_summary.csv",
    output_details_json="Results/evaluation_details.json",
    enable_weighted=True,
)
summary_df


✅ Summary saved: Results/evaluation_summary.csv
✅ Details saved: Results/evaluation_details.json


,file,mode,n_eval,accuracy,precision_true,recall_true,f1_true,confusion_matrix
0,deepseek_LIAR_no_subjects.csv,decision_agent,1233.0,0.502028,0.672340,0.227338,0.339785,"[[158, 537], [77, 461]]"
1,deepseek_LIAR_no_subjects.csv,expert__economy,1083.0,0.535549,0.695652,0.314239,0.432920,"[[192, 419], [84, 388]]"
2,deepseek_LIAR_no_subjects.csv,expert__health_science,1143.0,0.625547,0.630184,0.836391,0.718791,"[[547, 107], [321, 168]]"
3,deepseek_LIAR_no_subjects.csv,expert__politics,1089.0,0.453627,0.755556,0.055016,0.102564,"[[34, 584], [11, 460]]"
4,deepseek_LIAR_no_subjects.csv,majority_vote,1267.0,0.531176,0.694805,0.299720,0.418787,"[[214, 500], [94, 459]]"
...,...,...,...,...,...,...,...,...
104,llama_general_with_subjects.csv,expert__economy,1265.0,0.456126,0.662162,0.068820,0.124682,"[[49, 663], [25, 528]]"
105,llama_general_with_subjects.csv,expert__health_science,1148.0,0.493031,0.707143,0.154688,0.253846,"[[99, 541], [41, 467]]"
106,llama_general_with_subjects.csv,expert__politics,1249.0,0.485188,0.638614,0.184549,0.286349,"[[129, 570], [73, 477]]"
107,llama_general_with_subjects.csv,majority_vote,1267.0,0.472770,0.709091,0.109244,0.189320,"[[78, 636], [32, 521]]"
